# Tokenization and Normalization

The chapter opener showed a bookshop returning nothing for "car repair" because
"automobile repair" is stored under a different token. That failure begins at the
very first stage of the retrieval pipeline: how raw text is split into tokens and
normalized before any matching happens. This demo walks through that stage, from
the naive regex of Chapter 1 to a rule-based language detector, and ends by
observing the effect of normalization on a small BM25 index.

**Learning goals:**
- See exactly how the naive tokenizer breaks on realistic text
- Compare the naive regex, nltk, and spaCy tokenizers on the same sentence
- Apply the retrieval cleanup steps and the normalization transforms
- Handle text where word boundaries are not spaces (sub-word trigrams)
- Detect the language of a document with a handful of rules
- Measure how case folding changes a BM25 ranking

**Prerequisites:** Chapter 1 (feature extraction pipeline)

In [1]:
import re
import unicodedata
from unidecode import unidecode
import nltk
import spacy
from shared.display import print_table, display_md
from shared.text import stopwords_for
from shared.retrieval import document_frequencies, rank_collection_bm25

In [2]:
# One-time model/data downloads (quiet, safe to re-run)
nltk.download("punkt_tab", quiet=True)
nlp = spacy.load("en_core_web_sm")

## 1. The naive tokenizer and where it breaks

Chapter 1 introduced a one-line tokenizer: replace any run of non-word characters
(keeping hyphens) with a space, then split. It works on well-behaved prose. Give
it a sentence that exercises real text and it fails in several distinct ways at once.

In [3]:
def naive_tokenize(text: str) -> list[str]:
    text = re.sub(r'[^\w\-]+', ' ', text)
    return [t for t in text.split(' ') if t]

sentence = ("I buy my parents' 10% of U.K. startup for $1.4 billion. "
            "Dr. Watson's cat called Mrs. Hersley and it was w.r.o.n.g., more to come ...")

naive_tokens = naive_tokenize(sentence)
display_md(
    f"**Input:**\n\n> {sentence}\n\n"
    f"**Naive tokens ({len(naive_tokens)}):**\n\n`{naive_tokens}`"
)

**Input:**

> I buy my parents' 10% of U.K. startup for $1.4 billion. Dr. Watson's cat called Mrs. Hersley and it was w.r.o.n.g., more to come ...

**Naive tokens (31):**

`['I', 'buy', 'my', 'parents', '10', 'of', 'U', 'K', 'startup', 'for', '1', '4', 'billion', 'Dr', 'Watson', 's', 'cat', 'called', 'Mrs', 'Hersley', 'and', 'it', 'was', 'w', 'r', 'o', 'n', 'g', 'more', 'to', 'come']`

Four systematic problems appear at once:

- **Possessives depend on shape.** `parents'` collapses to `parents` (the apostrophe
  vanishes); `Watson's` splits into `Watson` and a stray `s`.
- **Numbers, currencies, percentages fall apart.** `10%` → `10`, `$1.4 billion` → `1`, `4`, `billion`.
- **Abbreviations disintegrate.** `U.K.`, `Dr.`, `Mrs.` break at their internal periods.
- **Punctuation vanishes.** Fine for retrieval, fatal for sentence analysis.

For a retrieval-only English pipeline this is survivable, but the distinctions are
gone for good.

## 2. Modern tokenization with nltk and spaCy

Both libraries ship word tokenizers built from rules plus abbreviation and exception
lists learned from corpora. Run all three on the same sentence and line them up.

In [4]:
naive_result = naive_tokenize(sentence)
nltk_result = nltk.word_tokenize(sentence)
spacy_result = [t.text for t in nlp(sentence)]

def pad(lst, n):
    return lst + [""] * (n - len(lst))

max_len = max(len(naive_result), len(nltk_result), len(spacy_result))
rows = list(zip(
    pad(naive_result, max_len),
    pad(nltk_result, max_len),
    pad(spacy_result, max_len),
))
print_table(rows, headers=["Naive (regex)", "NLTK", "spaCy"])

| Naive (regex)   | NLTK       | spaCy     |
|:----------------|:-----------|:----------|
| I               | I          | I         |
| buy             | buy        | buy       |
| my              | my         | my        |
| parents         | parents    | parents   |
| 10              | '          | '         |
| of              | 10         | 10        |
| U               | %          | %         |
| K               | of         | of        |
| startup         | U.K.       | U.K.      |
| for             | startup    | startup   |
| 1               | for        | for       |
| 4               | $          | $         |
| billion         | 1.4        | 1.4       |
| Dr              | billion    | billion   |
| Watson          | .          | .         |
| s               | Dr.        | Dr.       |
| cat             | Watson     | Watson    |
| called          | 's         | 's        |
| Mrs             | cat        | cat       |
| Hersley         | called     | called    |
| and             | Mrs.       | Mrs.      |
| it              | Hersley    | Hersley   |
| was             | and        | and       |
| w               | it         | it        |
| r               | was        | was       |
| o               | w.r.o.n.g. | w.r.o.n.g |
| n               | ,          | .         |
| g               | more       | ,         |
| more            | to         | more      |
| to              | come       | to        |
| come            | ...        | come      |
|                 |            | ...       |

In [5]:
display_md(
    f"**Token counts.** Naive: {len(naive_result)}, "
    f"NLTK: {len(nltk_result)}, spaCy: {len(spacy_result)}\n\n"
    "The trained tokenizers fix most of the naive failures:\n\n"
    "- **Numbers** stay whole (`1.4`), but `$` and `%` split off as their own tokens.\n"
    "- **Abbreviations** `U.K.`, `Dr.`, `Mrs.` are kept intact via curated lists.\n"
    "- **Possessives** keep the apostrophe visible; only `'s` carries a marker.\n\n"
    "nltk and spaCy agree on every token **except one**: nltk treats the artificial "
    "`w.r.o.n.g.` as a single abbreviation, spaCy splits its trailing period. On real "
    "text the two libraries almost always agree; novel abbreviations are where they diverge."
)

**Token counts.** Naive: 31, NLTK: 31, spaCy: 32

The trained tokenizers fix most of the naive failures:

- **Numbers** stay whole (`1.4`), but `$` and `%` split off as their own tokens.
- **Abbreviations** `U.K.`, `Dr.`, `Mrs.` are kept intact via curated lists.
- **Possessives** keep the apostrophe visible; only `'s` carries a marker.

nltk and spaCy agree on every token **except one**: nltk treats the artificial `w.r.o.n.g.` as a single abbreviation, spaCy splits its trailing period. On real text the two libraries almost always agree; novel abbreviations are where they diverge.

## 3. Retrieval cleanup

For retrieval we want a leaner list than any tokenizer produces. The standard cleanup
after tokenization is three filters, applied in order:

1. Drop single-letter tokens and pure punctuation.
2. Drop numbers and currency tokens (unless the collection is numeric).
3. Drop possessive `'s` tokens once the possessive has been used.

In [6]:
def is_punct(t):
    return all(unicodedata.category(c).startswith("P") or c in "$%" for c in t)

def cleanup(tokens):
    step1 = [t for t in tokens if len(t) > 1 and not is_punct(t)]
    step2 = [t for t in step1 if not re.fullmatch(r"[\d.,]+", t)]
    step3 = [t for t in step2 if t != "'s"]
    return step1, step2, step3

step1, step2, step3 = cleanup(nltk_result)

print_table(
    [
        ["0. spaCy/nltk tokens", len(nltk_result), " ".join(nltk_result)],
        ["1. drop 1-char + punct", len(step1), " ".join(step1)],
        ["2. drop numbers", len(step2), " ".join(step2)],
        ["3. drop possessive 's", len(step3), " ".join(step3)],
    ],
    headers=["Cleanup step", "Count", "Tokens"],
)

| Cleanup step           |   Count | Tokens                                                                                                                                     |
|:-----------------------|--------:|:-------------------------------------------------------------------------------------------------------------------------------------------|
| 0. spaCy/nltk tokens   |      31 | I buy my parents ' 10 % of U.K. startup for $ 1.4 billion . Dr. Watson 's cat called Mrs. Hersley and it was w.r.o.n.g. , more to come ... |
| 1. drop 1-char + punct |      24 | buy my parents 10 of U.K. startup for 1.4 billion Dr. Watson 's cat called Mrs. Hersley and it was w.r.o.n.g. more to come                 |
| 2. drop numbers        |      22 | buy my parents of U.K. startup for billion Dr. Watson 's cat called Mrs. Hersley and it was w.r.o.n.g. more to come                        |
| 3. drop possessive 's  |      21 | buy my parents of U.K. startup for billion Dr. Watson cat called Mrs. Hersley and it was w.r.o.n.g. more to come                           |

Each filter is cheap and reversible in principle, but once applied at ingestion the
information is gone from the index. The order matters: punctuation removal must run
before the possessive filter, or the bare `'` from `parents'` would linger.

## 4. Case, Unicode, and accents

A tokenizer separates words; normalization decides which surface forms of the *same*
word collapse to one token. Three transforms handle almost all European text. Take
one city name in three spellings:

In [7]:
variants = ["Zurich", "Zürich", "ZURICH"]

rows = []
for v in variants:
    case_folded = v.lower()
    nfkc = unicodedata.normalize("NFKC", case_folded)
    accent_folded = unidecode(nfkc)
    rows.append([v, case_folded, accent_folded])

print_table(rows, headers=["Original", "Case-folded", "+ Accent-folded"])

| Original   | Case-folded   | + Accent-folded   |
|:-----------|:--------------|:------------------|
| Zurich     | zurich        | zurich            |
| Zürich     | zürich        | zurich            |
| ZURICH     | zurich        | zurich            |

In [8]:
# NFKC canonicalises characters with multiple valid encodings
ligature = "ﬁle"            # 'fi' ligature, one codepoint
canonical = unicodedata.normalize("NFKC", ligature)

display_md(
    f"**Unicode normalization (NFKC)** canonicalises alternate encodings:\n\n"
    f"- `{ligature}` (len {len(ligature)}) → `{canonical}` (len {len(canonical)})\n\n"
    "After all three transforms, every spelling of the city collapses to `zurich`. "
    "A user can type any variant and match any document."
)

**Unicode normalization (NFKC)** canonicalises alternate encodings:

- `ﬁle` (len 3) → `file` (len 4)

After all three transforms, every spelling of the city collapses to `zurich`. A user can type any variant and match any document.

**Caution: accent folding is not free.** In German, "schon" (already) and "schön"
(beautiful) collide once folded; in French, "où" (where) and "ou" (or) become
identical. Fold accents only when the user population cannot easily type them (most
web search); keep them in single-language legal or scholarly indexes.

In [9]:
# The collision, made concrete
display_md(
    "**Accent-folding collisions:**\n\n"
    f"| Word | Meaning | Folded |\n|---|---|---|\n"
    f"| schön | beautiful (de) | {unidecode('schön')} |\n"
    f"| schon | already (de) | {unidecode('schon')} |\n"
    f"| où | where (fr) | {unidecode('où')} |\n"
    f"| ou | or (fr) | {unidecode('ou')} |\n\n"
    "Both German words now map to the same token: recall up, precision down."
)

**Accent-folding collisions:**

| Word | Meaning | Folded |
|---|---|---|
| schön | beautiful (de) | schon |
| schon | already (de) | schon |
| où | where (fr) | ou |
| ou | or (fr) | ou |

Both German words now map to the same token: recall up, precision down.

## 5. Sentence segmentation

Some tasks need whole sentences, not tokens: part-of-speech taggers work sentence by
sentence, and RAG systems group consecutive sentences into chunks. Splitting on `.`
fails on the same abbreviations we saw earlier.

In [10]:
text = ("Dr. Watson's cat called Mrs. Hersley. She was Egyptian. "
        "The cost was $1.5 million... Can you believe it?")

naive_split = [s.strip() for s in text.split(".") if s.strip()]
punkt_sents = nltk.sent_tokenize(text)
spacy_sents = [s.text.strip() for s in nlp(text).sents]

m = max(len(naive_split), len(punkt_sents), len(spacy_sents))
print_table(
    list(zip(pad(naive_split, m), pad(punkt_sents, m), pad(spacy_sents, m))),
    headers=["Naive split on '.'", "Punkt (nltk)", "spaCy"],
)

| Naive split on '.'      | Punkt (nltk)                                     | spaCy                                 |
|:------------------------|:-------------------------------------------------|:--------------------------------------|
| Dr                      | Dr. Watson's cat called Mrs. Hersley.            | Dr. Watson's cat called Mrs. Hersley. |
| Watson's cat called Mrs | She was Egyptian.                                | She was Egyptian.                     |
| Hersley                 | The cost was $1.5 million... Can you believe it? | The cost was $1.5 million...          |
| She was Egyptian        |                                                  | Can you believe it?                   |
| The cost was $1         |                                                  |                                       |
| 5 million               |                                                  |                                       |
| Can you believe it?     |                                                  |                                       |

In [11]:
display_md(
    f"Naive `.`-splitting produced {len(naive_split)} fragments; it breaks after "
    f"`Dr` and `Mrs` and inside `$1.5`. Punkt ({len(punkt_sents)} sentences) and spaCy "
    f"({len(spacy_sents)}) both carry abbreviation lists and keep those intact.\n\n"
    "Sentence segmentation is one of the few classical pipeline pieces that stayed "
    "useful into the LLM era: every RAG system still needs to decide where chunks begin."
)

Naive `.`-splitting produced 7 fragments; it breaks after `Dr` and `Mrs` and inside `$1.5`. Punkt (3 sentences) and spaCy (4) both carry abbreviation lists and keep those intact.

Sentence segmentation is one of the few classical pipeline pieces that stayed useful into the LLM era: every RAG system still needs to decide where chunks begin.

## 6. Word boundaries are not always spaces

Not every writing system uses spaces (Chinese, Japanese, Thai), and not every "word"
is one token (`QueryParser`, `word_tokenize`). When whole-word matching fails, index
overlapping **character trigrams** with a `#` marking word starts (Cavnar & Trenkle 1994).

In [12]:
def trigrams(text: str) -> list[str]:
    """Overlapping character trigrams; '#' marks a word boundary."""
    grams = []
    for word in text.lower().split():
        marked = "#" + word
        grams += [marked[i:i+3] for i in range(len(marked) - 2)]
    return grams

doc = "This course teaches multimedia retrieval."
query = "teach multtimedia"          # inflected form + a typo

doc_grams = set(trigrams(doc))
query_grams = trigrams(query)
matched = [g for g in query_grams if g in doc_grams]

display_md(
    f"**Document trigrams (sample):** `{trigrams(doc)[:8]} ...`\n\n"
    f"**Query:** `{query}`  (note: \"teach\" not \"teaches\", and a typo in \"multtimedia\")\n\n"
    f"**Query trigrams:** `{query_grams}`\n\n"
    f"**Matched {len(matched)} of {len(query_grams)}:** `{matched}`\n\n"
    "Even with a different inflection and a typo, most trigrams still match. This is the "
    "same principle behind BPE and WordPiece, which *learn* the sub-word vocabulary from a "
    "corpus instead of fixing it to trigrams (see the semantic-search chapter)."
)

**Document trigrams (sample):** `['#th', 'thi', 'his', '#co', 'cou', 'our', 'urs', 'rse'] ...`

**Query:** `teach multtimedia`  (note: "teach" not "teaches", and a typo in "multtimedia")

**Query trigrams:** `['#te', 'tea', 'eac', 'ach', '#mu', 'mul', 'ult', 'ltt', 'tti', 'tim', 'ime', 'med', 'edi', 'dia']`

**Matched 12 of 14:** `['#te', 'tea', 'eac', 'ach', '#mu', 'mul', 'ult', 'tim', 'ime', 'med', 'edi', 'dia']`

Even with a different inflection and a typo, most trigrams still match. This is the same principle behind BPE and WordPiece, which *learn* the sub-word vocabulary from a corpus instead of fixing it to trigrams (see the semantic-search chapter).

## 7. Language detection

Indexing and query understanding both depend on knowing the language. For long text a
few rules suffice: which script appears, which diacritics, and how many of each
language's stop words are present.

In [13]:
CHAR_SIGNALS = {"de": set("äöüß"), "fr": set("çàâêîôûéèùœ"), "es": set("ñ¿¡")}
STOPWORDS = {lang: stopwords_for(lang) for lang in ("en", "de", "fr")}

def detect_language(text: str) -> tuple[str, dict]:
    toks = re.findall(r"\w+", text.lower())
    scores = {}
    for lang, sw in STOPWORDS.items():
        stop_hits = sum(1 for t in toks if t in sw)
        char_hits = sum(1 for c in text.lower() if c in CHAR_SIGNALS.get(lang, set()))
        scores[lang] = stop_hits + 2 * char_hits
    best = max(scores, key=scores.get)
    # No stop word and no diacritic matched: the rules have no evidence to decide.
    if scores[best] == 0:
        return "undetermined", scores
    return best, scores

samples = [
    "Der schnelle braune Fuchs springt über den faulen Hund.",
    "The quick brown fox jumps over the lazy dog.",
    "Le renard brun rapide saute par-dessus le chien paresseux.",
    "Bücher von Goethe",
]
rows = []
for s in samples:
    lang, sc = detect_language(s)
    rows.append([s, sc["en"], sc["de"], sc["fr"], lang])
print_table(rows, headers=["Text", "en", "de", "fr", "Detected"])

| Text                                                       |   en |   de |   fr | Detected   |
|:-----------------------------------------------------------|-----:|-----:|-----:|:-----------|
| Der schnelle braune Fuchs springt über den faulen Hund.    |    0 |    5 |    0 | de         |
| The quick brown fox jumps over the lazy dog.               |    3 |    0 |    0 | en         |
| Le renard brun rapide saute par-dessus le chien paresseux. |    0 |    0 |    3 | fr         |
| Bücher von Goethe                                          |    0 |    3 |    0 | de         |

On long text the rules fire many times and the winner is clear. Short queries starve
them of evidence.

In [14]:
short = ["Mein computer", "pain", "ok"]
rows = []
for q in short:
    lang, sc = detect_language(q)
    rows.append([q, sc["en"], sc["de"], sc["fr"], lang])
print_table(rows, headers=["Query", "en", "de", "fr", "Detected"])

| Query         |   en |   de |   fr | Detected     |
|:--------------|-----:|-----:|-----:|:-------------|
| Mein computer |    0 |    1 |    0 | de           |
| pain          |    0 |    0 |    0 | undetermined |
| ok            |    0 |    0 |    0 | undetermined |

**Short queries break the rules.** "Mein computer" scrapes a single German stop word ("mein") and squeaks out a guess. "pain" (French for bread, English for suffering) and "ok" contain no stop word and no diacritic, so every language scores zero and the rules cannot decide. This regime needs a statistical classifier over character n-grams, developed with Naive Bayes in the intent-routing demo.

## 8. The payoff: normalization on a BM25 index

Everything above is preparation. Here is the effect on retrieval. Index a small
collection twice, once with the raw-case tokens and once case-folded, then run the
same lowercase query `woodland`. Several documents write it capitalised at a
sentence start (as in "Woodland Companions"), so the raw index never matches them.

In [15]:
from shared.synthetic_collection import MINI

query = ["woodland"]

# Two tokenizations of the same collection
raw_corpus = {d: naive_tokenize(t) for d, t in MINI.items()}
folded_corpus = {d: [w.lower() for w in naive_tokenize(t)] for d, t in MINI.items()}

raw_df = document_frequencies(raw_corpus)
folded_df = document_frequencies(folded_corpus)

display_md(
    "**Document frequency of the query term across schemes:**\n\n"
    f"| Term | Raw case | Case-folded |\n|---|---|---|\n"
    f"| woodland | {raw_df.get('woodland', 0)} | {folded_df.get('woodland', 0)} |\n"
    f"| Woodland | {raw_df.get('Woodland', 0)} | merged into `woodland` |\n\n"
    "Without folding, `woodland` and `Woodland` are different terms, so the lowercase "
    "query reaches only the documents that happen to spell it in lowercase."
)

**Document frequency of the query term across schemes:**

| Term | Raw case | Case-folded |
|---|---|---|
| woodland | 2 | 4 |
| Woodland | 2 | merged into `woodland` |

Without folding, `woodland` and `Woodland` are different terms, so the lowercase query reaches only the documents that happen to spell it in lowercase.

In [16]:
raw_ranking = rank_collection_bm25(query, raw_corpus, raw_df)
folded_ranking = rank_collection_bm25(query, folded_corpus, folded_df)

def fmt(ranking, n=6):
    return [f"{d} ({s:.2f})" for d, s in ranking[:n] if s > 0]

r, f = fmt(raw_ranking), fmt(folded_ranking)
m = max(len(r), len(f))
print_table(
    list(zip(pad(r, m), pad(f, m))),
    headers=["Raw case, top hits", "Case-folded, top hits"],
)

| Raw case, top hits   | Case-folded, top hits   |
|:---------------------|:------------------------|
| b3 (1.46)            | b2 (0.67)               |
| b1 (1.34)            | b3 (0.65)               |
|                      | b11 (0.63)              |
|                      | b1 (0.59)               |

For the query `woodland`, case folding brings the documents that spell it `Woodland`
(capitalised at a sentence start) into the result set; the raw index misses them
entirely. One normalization step, measurable recall gain, and this is before any
stemming, which the next demo adds.

## Summary

| Tokenizer | Strength | Weakness |
| --- | --- | --- |
| Naive regex | Fast, zero dependencies | Breaks on punctuation, numbers, abbreviations |
| nltk word_tokenize | Curated abbreviation lists | English-centric |
| spaCy | Part of a full linguistic pipeline | Heavier to load |
| Trigram shingling | Handles no-space scripts, typos | Not human-readable |

| Normalization | Recall gain | Precision loss | Risk |
| --- | --- | --- | --- |
| Case folding | High | Low | Collapses proper nouns |
| Unicode (NFKC) | Essential | None | Always apply |
| Accent folding | Medium | Medium | Language-dependent collisions |

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Takeaway</strong><br>
There is no universally correct tokenizer or normalization. Each choice trades recall for precision, propagates through the whole pipeline, and is expensive to change once the index is built.
</div>

## Try it yourself

1. Add a sentence to Section 2 where all three tokenizers disagree.
2. In Section 8, add accent folding to `folded_corpus` and invent a query that only
   matches after folding.
3. Extend `detect_language` with a fourth language. What breaks on short queries?

In [17]:
my_text = "Replace this with your own tricky text!"
display_md(
    f"**Naive:** `{naive_tokenize(my_text)}`\n\n"
    f"**NLTK:** `{nltk.word_tokenize(my_text)}`\n\n"
    f"**spaCy:** `{[t.text for t in nlp(my_text)]}`\n\n"
    f"**Detected language:** {detect_language(my_text)[0]}"
)

**Naive:** `['Replace', 'this', 'with', 'your', 'own', 'tricky', 'text']`

**NLTK:** `['Replace', 'this', 'with', 'your', 'own', 'tricky', 'text', '!']`

**spaCy:** `['Replace', 'this', 'with', 'your', 'own', 'tricky', 'text', '!']`

**Detected language:** en